# Limpeza dos dados - Eleições presidenciais 2014 a 2026

Neste notebook eu preparo o arquivo `eleicoes_presidente.csv` para a análise e para o dashboard.
Cada linha do arquivo mostra os votos de um candidato em uma zona eleitoral, em um turno.

A limpeza aqui é **a mesma feita no Power Query** do Power BI (veja `powerbi/guia_dashboard.md`).
Os dois caminhos geram as mesmas 5 tabelas, com os mesmos nomes de colunas:

| Tabela | Uma linha = |
|---|---|
| `dTurno` | uma eleição e turno |
| `dCandidato` | um candidato em uma eleição |
| `dMunicipio` | um município (ou cidade do exterior) |
| `fVotos` | votos de um candidato em um município, em um turno |
| `fComparecimento` | eleitorado e comparecimento de um município, em um turno |

**Etapas**

1. Configuração
2. Leitura e visão geral
3. Conferência com os totais oficiais
4. Limpeza
5. Criação das tabelas
6. Conferência final e exportação

---
## 1. Configuração

In [1]:
caminho_dados = '../data/eleicoes_presidente.csv'
pasta_saida = '../data/limpo'

In [2]:
import os

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## 2. Leitura e visão geral

In [3]:
# Lendo os dados
# low_memory=False faz o pandas ler o arquivo inteiro antes de decidir o tipo de cada coluna
df_eleicoes = pd.read_csv(caminho_dados, sep=';', low_memory=False)

df_eleicoes.head()

,ano,nr_turno,sg_regiao,sg_uf,cd_municipio_tse,cd_municipio_ibge,municipio,nr_zona,nr_candidato,nm_urna,sg_partido,situacao_turno,qt_votos_nominais,zona_aptos,zona_comparecimento,zona_abstencoes,zona_votos_validos,zona_votos_brancos,zona_votos_nulos,populacao_2022,alfabetizacao_2022,pib_mil_reais
0,2014,1,N,AC,1120,1200013.00,Acrelândia,8,13,DILMA,PT,2º TURNO,2293,9143,6999,2144,6550,108,341,14021.00,88.35,207822.00
1,2014,1,N,AC,1120,1200013.00,Acrelândia,8,16,ZÉ MARIA,PSTU,NÃO ELEITO,3,9143,6999,2144,6550,108,341,14021.00,88.35,207822.00
2,2014,1,N,AC,1120,1200013.00,Acrelândia,8,20,PASTOR EVERALDO,PSC,NÃO ELEITO,19,9143,6999,2144,6550,108,341,14021.00,88.35,207822.00
3,2014,1,N,AC,1120,1200013.00,Acrelândia,8,21,MAURO IASI,PCB,NÃO ELEITO,2,9143,6999,2144,6550,108,341,14021.00,88.35,207822.00
4,2014,1,N,AC,1120,1200013.00,Acrelândia,8,27,EYMAEL,PSDC,NÃO ELEITO,13,9143,6999,2144,6550,108,341,14021.00,88.35,207822.00


In [4]:
print('Linhas:', df_eleicoes.shape[0])
print('Colunas:', df_eleicoes.shape[1])

Linhas: 340702
Colunas: 22


In [5]:
df_eleicoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 340702 entries, 0 to 340701
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ano                  340702 non-null  int64  
 1   nr_turno             340702 non-null  int64  
 2   sg_regiao            332438 non-null  str    
 3   sg_uf                340702 non-null  str    
 4   cd_municipio_tse     340702 non-null  int64  
 5   cd_municipio_ibge    332438 non-null  float64
 6   municipio            340702 non-null  str    
 7   nr_zona              340702 non-null  int64  
 8   nr_candidato         340702 non-null  int64  
 9   nm_urna              340702 non-null  str    
 10  sg_partido           340702 non-null  str    
 11  situacao_turno       340702 non-null  str    
 12  qt_votos_nominais    340702 non-null  int64  
 13  zona_aptos           340702 non-null  int64  
 14  zona_comparecimento  340702 non-null  int64  
 15  zona_abstencoes      340702 

In [6]:
# Contando valores vazios por coluna
df_eleicoes.isna().sum()

ano                       0
nr_turno                  0
sg_regiao              8264
sg_uf                     0
cd_municipio_tse          0
cd_municipio_ibge      8264
municipio                 0
nr_zona                   0
nr_candidato              0
nm_urna                   0
sg_partido                0
situacao_turno            0
qt_votos_nominais         0
zona_aptos                0
zona_comparecimento       0
zona_abstencoes           0
zona_votos_validos        0
zona_votos_brancos        0
zona_votos_nulos          0
populacao_2022         8278
alfabetizacao_2022     8278
pib_mil_reais          8278
dtype: int64

In [7]:
# Verificando linhas repetidas
print('Linhas duplicadas:', df_eleicoes.duplicated().sum())

Linhas duplicadas: 0


---
## 3. Conferência com os totais oficiais

Antes de limpar, confiro se o arquivo bate com o resultado oficial do TSE.

**Cuidado:** as colunas que começam com `zona_` (aptos, comparecimento, abstenções, brancos, nulos)
se repetem em todas as linhas de candidato da mesma zona. Se eu somar direto, o total fica
multiplicado pelo número de candidatos. Por isso, primeiro fico com uma linha por zona.

In [8]:
# Ficando com uma linha por zona em cada turno
df_zonas = df_eleicoes.drop_duplicates(subset=['ano', 'nr_turno', 'cd_municipio_tse', 'nr_zona'])

print('Linhas no arquivo:', len(df_eleicoes))
print('Zonas (sem repetição):', len(df_zonas))

Linhas no arquivo: 340702
Zonas (sem repetição): 50422


In [9]:
# Totais oficiais do TSE
df_oficial = pd.DataFrame({
    'ano': [2014, 2014, 2018, 2018, 2022, 2022, 2026, 2026],
    'nr_turno': [1, 2, 1, 2, 1, 2, 1, 2],
    'comparecimento_oficial': [115122883, 112683879, 117364654, 115933451, 123682372, 124252796, 125275835, 128236074],
    'votos_validos_oficial': [104023802, 105542273, 107050749, 104838753, 118229719, 118552353, 119300788, 124082273],
})

In [10]:
# Somando o comparecimento e os votos válidos de cada turno e comparando com o oficial
df_totais = df_zonas.groupby(['ano', 'nr_turno'])[['zona_comparecimento', 'zona_votos_validos']].sum().reset_index()

df_conferencia = df_totais.merge(df_oficial, on=['ano', 'nr_turno'])
df_conferencia['comparecimento_confere'] = df_conferencia['zona_comparecimento'] == df_conferencia['comparecimento_oficial']
df_conferencia['validos_confere'] = df_conferencia['zona_votos_validos'] == df_conferencia['votos_validos_oficial']
df_conferencia

,ano,nr_turno,zona_comparecimento,zona_votos_validos,comparecimento_oficial,votos_validos_oficial,comparecimento_confere,validos_confere
0,2014,1,115122883,104023802,115122883,104023802,True,True
1,2014,2,112683879,105542273,112683879,105542273,True,True
2,2018,1,117364654,107050749,117364654,107050749,True,True
3,2018,2,115933451,104838753,115933451,104838753,True,True
4,2022,1,123682372,118229719,123682372,118229719,True,True
5,2022,2,124252796,118552353,124252796,118552353,True,True
6,2026,1,125275835,119300788,125275835,119300788,True,True
7,2026,2,128236074,124082273,128236074,124082273,True,True


Para ver o tamanho do erro, somo o comparecimento **sem** tirar as repetições:

In [11]:
# Forma errada: somando direto no arquivo completo
df_eleicoes.groupby(['ano', 'nr_turno'])['zona_comparecimento'].sum()

ano   nr_turno
2014  1           1238634927
      2            225367751
2018  1           1525740502
      2            231866902
2022  1           1360506092
      2            248505592
2026  1           1503310020
      2            256472148
Name: zona_comparecimento, dtype: int64

---
## 4. Limpeza

Cada passo abaixo tem um passo igual no Power Query (o nome do passo está entre parênteses).

### 4.1 Tipos (passo `Tipos`)

A coluna `cd_municipio_ibge` foi lida como número decimal (ex.: `1200013.0`) porque tem valores
vazios. O tipo `Int64` (com I maiúsculo) guarda número inteiro e aceita vazio. O mesmo vale para
a população.

In [12]:
# Ajustando colunas de código e população para número inteiro
df_eleicoes['cd_municipio_ibge'] = df_eleicoes['cd_municipio_ibge'].astype('Int64')
df_eleicoes['populacao_2022'] = df_eleicoes['populacao_2022'].astype('Int64')

df_eleicoes[['cd_municipio_ibge', 'populacao_2022']].dtypes

cd_municipio_ibge    Int64
populacao_2022       Int64
dtype: object

### 4.2 Espaços nos textos (passo `Textos`)

In [13]:
# Tirando espaços do começo e do fim dos textos
df_eleicoes['municipio'] = df_eleicoes['municipio'].str.strip()
df_eleicoes['nm_urna'] = df_eleicoes['nm_urna'].str.strip()
df_eleicoes['sg_partido'] = df_eleicoes['sg_partido'].str.strip()
df_eleicoes['sg_uf'] = df_eleicoes['sg_uf'].str.strip()
df_eleicoes['situacao_turno'] = df_eleicoes['situacao_turno'].str.strip()

### 4.3 Região e UF por extenso (passos `Regiao`, `NomeRegiao` e `NomeUF`)

Os vazios de `sg_regiao` são todos do **exterior** (`sg_uf = 'ZZ'`). Preencho com `EX` e crio
colunas com o nome da região e do estado, que ficam melhores nos gráficos e no mapa.

In [14]:
# Ajustando valores vazios na coluna de região
df_eleicoes['sg_regiao'] = df_eleicoes['sg_regiao'].fillna('EX')

df_eleicoes['sg_regiao'].value_counts()

sg_regiao
SE    107026
NE    103099
S      68011
CO     28218
N      26084
EX      8264
Name: count, dtype: int64

In [15]:
# Criando coluna com o nome da região
nomes_regioes = {
    'N': 'Norte', 'NE': 'Nordeste', 'SE': 'Sudeste', 'S': 'Sul', 'CO': 'Centro-Oeste', 'EX': 'Exterior',
}
df_eleicoes['nm_regiao'] = df_eleicoes['sg_regiao'].map(nomes_regioes)

# Criando coluna com o nome do estado
nomes_ufs = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas', 'BA': 'Bahia', 'CE': 'Ceará',
    'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão',
    'MT': 'Mato Grosso', 'MS': 'Mato Grosso do Sul', 'MG': 'Minas Gerais', 'PA': 'Pará',
    'PB': 'Paraíba', 'PR': 'Paraná', 'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro',
    'RN': 'Rio Grande do Norte', 'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima',
    'SC': 'Santa Catarina', 'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins', 'ZZ': 'Exterior',
}
df_eleicoes['nm_uf'] = df_eleicoes['sg_uf'].map(nomes_ufs)

df_eleicoes[['sg_regiao', 'nm_regiao', 'sg_uf', 'nm_uf']].drop_duplicates().head()

,sg_regiao,nm_regiao,sg_uf,nm_uf
0,N,Norte,AC,Acre
240,NE,Nordeste,AL,Alagoas
1291,N,Norte,AM,Amazonas
2010,N,Norte,AP,Amapá
2177,NE,Nordeste,BA,Bahia


### 4.4 Exterior e municípios com o mesmo nome (passos `Exterior` e `MunicipioUF`)

Crio uma coluna que marca o exterior, para filtrar nos mapas e nos indicadores. Também crio o nome
do município junto da UF, porque vários municípios têm o mesmo nome em estados diferentes.

In [16]:
# Criando coluna que marca o exterior
df_eleicoes['exterior'] = df_eleicoes['sg_uf'] == 'ZZ'

# Criando coluna com município e UF
df_eleicoes['municipio_uf'] = df_eleicoes['municipio'] + ' (' + df_eleicoes['sg_uf'] + ')'

df_eleicoes['exterior'].value_counts()

exterior
False    332438
True       8264
Name: count, dtype: int64

In [17]:
# Exemplo: nomes de município que aparecem em mais de uma UF
df_municipios_br = df_eleicoes[df_eleicoes['exterior'] == False].drop_duplicates(subset='cd_municipio_tse')
df_municipios_br['municipio'].value_counts().head()

municipio
São Domingos    5
Bom Jesus       5
Bonito          4
Planalto        4
Santa Inês      4
Name: count, dtype: int64

### 4.5 Partido que mudou de sigla (passo `PartidoAtual`)

Em 2017 o PSDC passou a se chamar DC. É o mesmo partido, então crio uma coluna com a sigla atual
para comparar as eleições. A coluna original continua igual.

In [18]:
# Criando coluna com a sigla atual do partido
df_eleicoes['sg_partido_atual'] = df_eleicoes['sg_partido'].replace({'PSDC': 'DC'})

df_eleicoes[df_eleicoes['sg_partido'] != df_eleicoes['sg_partido_atual']][['ano', 'sg_partido', 'sg_partido_atual']].drop_duplicates()

,ano,sg_partido,sg_partido_atual
4,2014,PSDC,DC


### 4.6 Chaves (passos `ChaveTurno` e `ChaveCandidato`)

O mesmo número de candidato se repete em eleições diferentes (o 13 existe em todas). Para ligar as
tabelas, crio chaves de texto que juntam o ano com o turno e o ano com o número do candidato.

In [19]:
# Criando as chaves
df_eleicoes['chave_turno'] = df_eleicoes['ano'].astype(str) + '-' + df_eleicoes['nr_turno'].astype(str)
df_eleicoes['chave_candidato'] = df_eleicoes['ano'].astype(str) + '-' + df_eleicoes['nr_candidato'].astype(str)

df_eleicoes[['ano', 'nr_turno', 'nr_candidato', 'chave_turno', 'chave_candidato']].head()

,ano,nr_turno,nr_candidato,chave_turno,chave_candidato
0,2014,1,13,2014-1,2014-13
1,2014,1,16,2014-1,2014-16
2,2014,1,20,2014-1,2014-20
3,2014,1,21,2014-1,2014-21
4,2014,1,27,2014-1,2014-27


### Valores vazios que ficam

Os indicadores do IBGE continuam vazios em dois casos, e **não preencho**:

- **Exterior**: não existe dado do IBGE para cidades fora do Brasil.
- **Boa Esperança do Norte (MT)**: município criado depois do Censo 2022.

Inventar população ou PIB para um lugar sem esse dado mudaria o resultado da análise.

In [20]:
# Município do Brasil sem dados do Censo
sem_populacao = df_eleicoes[df_eleicoes['populacao_2022'].isna() & (df_eleicoes['exterior'] == False)]
sem_populacao[['ano', 'municipio', 'sg_uf']].drop_duplicates()

,ano,municipio,sg_uf
282194,2026,Boa Esperança do Norte,MT


---
## 5. Criação das tabelas

### dTurno

In [21]:
# Uma linha por eleição e turno
dTurno = df_eleicoes[['chave_turno', 'ano', 'nr_turno']].drop_duplicates()

# Criando o rótulo usado nos filtros do dashboard e a ordem dos turnos
dTurno['rotulo_turno'] = dTurno['ano'].astype(str) + ' · ' + dTurno['nr_turno'].astype(str) + 'º turno'
dTurno['ordem_turno'] = dTurno['ano'] * 10 + dTurno['nr_turno']

dTurno = dTurno.sort_values('ordem_turno').reset_index(drop=True)
dTurno

,chave_turno,ano,nr_turno,rotulo_turno,ordem_turno
0,2014-1,2014,1,2014 · 1º turno,20141
1,2014-2,2014,2,2014 · 2º turno,20142
2,2018-1,2018,1,2018 · 1º turno,20181
3,2018-2,2018,2,2018 · 2º turno,20182
4,2022-1,2022,1,2022 · 1º turno,20221
5,2022-2,2022,2,2022 · 2º turno,20222
6,2026-1,2026,1,2026 · 1º turno,20261
7,2026-2,2026,2,2026 · 2º turno,20262


### dCandidato

In [22]:
# Uma linha por candidato em cada eleição
dCandidato = df_eleicoes[['chave_candidato', 'ano', 'nr_candidato', 'nm_urna', 'sg_partido', 'sg_partido_atual']].drop_duplicates()
dCandidato = dCandidato.sort_values(['ano', 'nr_candidato']).reset_index(drop=True)

print('Candidatos:', len(dCandidato))
dCandidato.head()

Candidatos: 47


,chave_candidato,ano,nr_candidato,nm_urna,sg_partido,sg_partido_atual
0,2014-13,2014,13,DILMA,PT,PT
1,2014-16,2014,16,ZÉ MARIA,PSTU,PSTU
2,2014-20,2014,20,PASTOR EVERALDO,PSC,PSC
3,2014-21,2014,21,MAURO IASI,PCB,PCB
4,2014-27,2014,27,EYMAEL,PSDC,DC


### dMunicipio

O PIB muda a cada ano. Para descrever o perfil de cada município, uso o PIB **mais recente**
(2023, ou 2022 quando o de 2023 não foi publicado), que está nas linhas de 2022 e 2026.

In [23]:
# Uma linha por município
colunas_municipio = ['cd_municipio_tse', 'cd_municipio_ibge', 'municipio', 'municipio_uf', 'sg_uf', 'nm_uf',
                     'sg_regiao', 'nm_regiao', 'exterior', 'populacao_2022', 'alfabetizacao_2022']
dMunicipio = df_eleicoes[colunas_municipio].drop_duplicates(subset='cd_municipio_tse')

# PIB mais recente de cada município
pib_recente = df_eleicoes[df_eleicoes['ano'] >= 2022][['cd_municipio_tse', 'pib_mil_reais']].drop_duplicates(subset='cd_municipio_tse')
dMunicipio = dMunicipio.merge(pib_recente, on='cd_municipio_tse', how='left')

# Criando coluna de PIB per capita (o PIB está em mil reais)
dMunicipio['pib_per_capita'] = dMunicipio['pib_mil_reais'] * 1000 / dMunicipio['populacao_2022']

dMunicipio[['municipio_uf', 'populacao_2022', 'pib_mil_reais', 'pib_per_capita']].head()

,municipio_uf,populacao_2022,pib_mil_reais,pib_per_capita
0,Acrelândia (AC),14021,502663.00,35850.72
1,Assis Brasil (AC),8100,187366.00,23131.60
2,Brasiléia (AC),26000,929800.00,35761.54
3,Bujari (AC),12917,365419.00,28289.77
4,Capixaba (AC),10392,530657.00,51063.99


Faixas de tamanho do município. Uso uma função com `if`, igual à **coluna condicional** do Power Query.
A coluna de ordem serve para o Power BI mostrar as faixas da menor para a maior.

In [24]:
def faixa_populacao(populacao):
    if pd.isna(populacao):
        return 'Sem dado'
    elif populacao <= 5000:
        return 'Até 5 mil'
    elif populacao <= 20000:
        return '5 a 20 mil'
    elif populacao <= 100000:
        return '20 a 100 mil'
    elif populacao <= 500000:
        return '100 a 500 mil'
    else:
        return 'Mais de 500 mil'


ordem_faixas = {'Sem dado': 0, 'Até 5 mil': 1, '5 a 20 mil': 2, '20 a 100 mil': 3, '100 a 500 mil': 4, 'Mais de 500 mil': 5}

dMunicipio['faixa_populacao'] = dMunicipio['populacao_2022'].apply(faixa_populacao)
dMunicipio['ordem_faixa_populacao'] = dMunicipio['faixa_populacao'].map(ordem_faixas)

dMunicipio['faixa_populacao'].value_counts()

faixa_populacao
5 a 20 mil         2537
20 a 100 mil       1390
Até 5 mil          1324
100 a 500 mil       278
Sem dado            196
Mais de 500 mil      41
Name: count, dtype: int64

In [25]:
# Endereço usado pelo mapa do Power BI
def local_mapa(linha):
    if linha['exterior']:
        return linha['municipio']
    else:
        return linha['municipio'] + ', ' + linha['nm_uf'] + ', Brasil'


dMunicipio['local_mapa'] = dMunicipio.apply(local_mapa, axis=1)

dMunicipio = dMunicipio.sort_values('cd_municipio_tse').reset_index(drop=True)
print('Municípios:', len(dMunicipio))
dMunicipio.head()

Municípios: 5766


,cd_municipio_tse,cd_municipio_ibge,municipio,municipio_uf,sg_uf,nm_uf,sg_regiao,nm_regiao,exterior,populacao_2022,alfabetizacao_2022,pib_mil_reais,pib_per_capita,faixa_populacao,ordem_faixa_populacao,local_mapa
0,19,1100106,Guajará-Mirim,Guajará-Mirim (RO),RO,Rondônia,N,Norte,False,39387,94.16,1145034.00,29071.37,20 a 100 mil,3,"Guajará-Mirim, Rondônia, Brasil"
1,35,1100205,Porto Velho,Porto Velho (RO),RO,Rondônia,N,Norte,False,460434,95.64,25402193.00,55170.11,100 a 500 mil,4,"Porto Velho, Rondônia, Brasil"
2,51,1100122,Ji-Paraná,Ji-Paraná (RO),RO,Rondônia,N,Norte,False,124333,94.37,5824472.00,46845.74,100 a 500 mil,4,"Ji-Paraná, Rondônia, Brasil"
3,78,1100023,Ariquemes,Ariquemes (RO),RO,Rondônia,N,Norte,False,96833,94.08,4383815.00,45271.91,20 a 100 mil,3,"Ariquemes, Rondônia, Brasil"
4,94,1100049,Cacoal,Cacoal (RO),RO,Rondônia,N,Norte,False,86887,93.71,3848468.00,44292.79,20 a 100 mil,3,"Cacoal, Rondônia, Brasil"


### fVotos

Somo os votos das zonas para ter uma linha por município, turno e candidato.

In [26]:
# Somando os votos de cada candidato no município
fVotos = (df_eleicoes
    .groupby(['chave_turno', 'ano', 'nr_turno', 'cd_municipio_tse', 'chave_candidato', 'situacao_turno'])
    ['qt_votos_nominais'].sum()
    .reset_index())

print('Linhas:', len(fVotos))
fVotos.head()

Linhas: 309308


,chave_turno,ano,nr_turno,cd_municipio_tse,chave_candidato,situacao_turno,qt_votos_nominais
0,2014-1,2014,1,19,2014-13,2º TURNO,8701
1,2014-1,2014,1,19,2014-16,NÃO ELEITO,7
2,2014-1,2014,1,19,2014-20,NÃO ELEITO,261
3,2014-1,2014,1,19,2014-21,NÃO ELEITO,5
4,2014-1,2014,1,19,2014-27,NÃO ELEITO,7


### fComparecimento

Primeiro fico com uma linha por zona (para não somar o mesmo total várias vezes) e depois somo as
zonas de cada município. Os nomes das colunas perdem o prefixo `zona_`.

In [27]:
# Uma linha por zona
df_zonas = df_eleicoes.drop_duplicates(subset=['chave_turno', 'cd_municipio_tse', 'nr_zona'])

# Somando as zonas de cada município
fComparecimento = (df_zonas
    .groupby(['chave_turno', 'ano', 'nr_turno', 'cd_municipio_tse'])
    .agg(
        aptos=('zona_aptos', 'sum'),
        comparecimento=('zona_comparecimento', 'sum'),
        abstencoes=('zona_abstencoes', 'sum'),
        votos_validos=('zona_votos_validos', 'sum'),
        votos_brancos=('zona_votos_brancos', 'sum'),
        votos_nulos=('zona_votos_nulos', 'sum'),
    )
    .reset_index())

print('Linhas:', len(fComparecimento))
fComparecimento.head()

Linhas: 45834


,chave_turno,ano,nr_turno,cd_municipio_tse,aptos,comparecimento,abstencoes,votos_validos,votos_brancos,votos_nulos
0,2014-1,2014,1,19,26189,21598,4591,20377,379,842
1,2014-1,2014,1,35,302773,250698,52075,231885,6878,11935
2,2014-1,2014,1,51,85624,65328,20296,61331,1421,2576
3,2014-1,2014,1,78,58991,49508,9483,46421,1184,1903
4,2014-1,2014,1,94,60227,45235,14992,42267,1087,1881


---
## 6. Conferência final e exportação

As tabelas novas precisam continuar batendo com os totais oficiais da etapa 3.

In [28]:
# Totais por turno nas tabelas novas
votos_turno = fVotos.groupby(['ano', 'nr_turno'])['qt_votos_nominais'].sum().reset_index()
comparecimento_turno = fComparecimento.groupby(['ano', 'nr_turno'])[['comparecimento', 'votos_validos']].sum().reset_index()

df_conferencia_final = comparecimento_turno.merge(votos_turno, on=['ano', 'nr_turno']).merge(df_oficial, on=['ano', 'nr_turno'])
df_conferencia_final['confere'] = (
    (df_conferencia_final['comparecimento'] == df_conferencia_final['comparecimento_oficial'])
    & (df_conferencia_final['votos_validos'] == df_conferencia_final['votos_validos_oficial'])
    & (df_conferencia_final['qt_votos_nominais'] == df_conferencia_final['votos_validos_oficial'])
)
df_conferencia_final

,ano,nr_turno,comparecimento,votos_validos,qt_votos_nominais,comparecimento_oficial,votos_validos_oficial,confere
0,2014,1,115122883,104023802,104023802,115122883,104023802,True
1,2014,2,112683879,105542273,105542273,112683879,105542273,True
2,2018,1,117364654,107050749,107050749,117364654,107050749,True
3,2018,2,115933451,104838753,104838753,115933451,104838753,True
4,2022,1,123682372,118229719,118229719,123682372,118229719,True
5,2022,2,124252796,118552353,118552353,124252796,118552353,True
6,2026,1,125275835,119300788,119300788,125275835,119300788,True
7,2026,2,128236074,124082273,124082273,128236074,124082273,True


In [29]:
# Conferindo se as chaves das tabelas são únicas
print('dTurno sem chave repetida:', dTurno['chave_turno'].is_unique)
print('dCandidato sem chave repetida:', dCandidato['chave_candidato'].is_unique)
print('dMunicipio sem chave repetida:', dMunicipio['cd_municipio_tse'].is_unique)

dTurno sem chave repetida: True
dCandidato sem chave repetida: True
dMunicipio sem chave repetida: True


In [30]:
# Salvando as tabelas
os.makedirs(pasta_saida, exist_ok=True)

dTurno.to_csv(f'{pasta_saida}/dTurno.csv', sep=';', index=False)
dCandidato.to_csv(f'{pasta_saida}/dCandidato.csv', sep=';', index=False)
dMunicipio.to_csv(f'{pasta_saida}/dMunicipio.csv', sep=';', index=False)
fVotos.to_csv(f'{pasta_saida}/fVotos.csv', sep=';', index=False)
fComparecimento.to_csv(f'{pasta_saida}/fComparecimento.csv', sep=';', index=False)

print('Tabelas salvas em', pasta_saida)

Tabelas salvas em ../data/limpo


---
## Resumo das decisões

| Situação | O que fiz | Por quê |
|---|---|---|
| Colunas `zona_` repetidas em cada candidato | Tirei as repetições antes de somar | Somar direto multiplica o total |
| Código IBGE e população lidos como decimal | Converti para `Int64` | São números inteiros; `Int64` aceita vazio |
| Região vazia no exterior | Preenchi com `EX` e criei `nm_regiao` e `nm_uf` | Rótulo legível; não muda nenhum número |
| Indicadores vazios no exterior e em Boa Esperança do Norte | Não preenchi; criei a coluna `exterior` | Não existe dado do IBGE para esses lugares |
| Municípios com o mesmo nome | Criei `municipio_uf` e uso sempre o código | Evita misturar cidades diferentes |
| PSDC virou DC | Criei `sg_partido_atual` | Mesmo partido com outra sigla |
| Número de candidato repete entre eleições | Criei `chave_turno` e `chave_candidato` | Ligar as tabelas sem misturar anos |
| PIB muda a cada ano | Usei o mais recente (2023) para o perfil do município | Um valor por município, comparável entre todos |
| Votos e totais por zona | Somei por município | É o nível usado nas perguntas e no dashboard |

## Limitações

- O exterior não tem dados do IBGE.
- Boa Esperança do Norte (MT) não tem dados do Censo 2022.
- O 2º turno de 2026 (25/10/2026) usa os arquivos publicados pelo TSE em 28/10/2026; se o TSE
  republicar os arquivos, rode o fluxo de novo para atualizar.
- Em 2014 e 2018 faltam 684 e 470 eleitores aptos de zonas do exterior onde ninguém votou.
- O PIB per capita usa a população de 2022.
- Os indicadores descrevem o município, não o eleitor: uma relação entre PIB e voto não mostra
  como cada pessoa votou.